# AI013 — Time Series Forecasting: Testing & Analysis Notebook

**Task:** AI013 - Time Series Forecasting  
**Author:** Divjot  
**Branch:** ai-ml/ai013/divjot-forecasting  

This notebook covers:
1. Loading the trained model
2. Generating predictions
3. Predicted vs actual comparison
4. Trend analysis
5. Saving outputs and reports

## 1. Setup & Imports

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Add forecasting module to path
sys.path.append(str(Path('../../src/model/forecasting').resolve()))

from dataset_builder import build_dataset
from sequence_generator import create_sequences, train_test_split_sequences
from model import ForecastingModel
from predictor import predict, save_predictions, evaluate

print('All modules loaded successfully.')

## 2. Load Data and Sequences

In [ ]:
DATA_PATH = Path('../../cleaning/data/output/cleaned_data.csv').resolve()
MODEL_PATH = Path('../../models/ai013_forecasting/forecasting_model.npz').resolve()

df = build_dataset(str(DATA_PATH))
X, y = create_sequences(df, target_col='severity_score', sequence_length=5)
X_train, X_test, y_train, y_test = train_test_split_sequences(X, y, test_ratio=0.2)

print(f'Test sequences: {len(X_test)}')
print(f'Test targets:   {len(y_test)}')

## 3. Load Trained Model and Generate Predictions

In [ ]:
model = ForecastingModel(input_size=5, hidden_size=16)
model.load(str(MODEL_PATH))

y_pred = predict(model, X_test)

print(f'Predictions generated: {len(y_pred)}')
print(f'\nFirst 5 predictions: {y_pred[:5]}')
print(f'First 5 actuals:     {y_test[:5]}')

## 4. Predicted vs Actual Comparison

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(y_test, label='Actual', marker='o', linewidth=2, color='blue')
plt.plot(y_pred, label='Predicted', marker='x', linewidth=2, linestyle='--', color='red')
plt.title('Predicted vs Actual Severity Score')
plt.xlabel('Test Sample Index')
plt.ylabel('Severity Score')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Error distribution
errors = y_pred - y_test

plt.figure(figsize=(10, 4))
plt.bar(range(len(errors)), errors, color=['red' if e > 0 else 'blue' for e in errors])
plt.axhline(0, color='black', linewidth=1)
plt.title('Prediction Error (Predicted - Actual)')
plt.xlabel('Test Sample Index')
plt.ylabel('Error')
plt.tight_layout()
plt.show()

## 5. Evaluation Metrics

In [ ]:
metrics = evaluate(y_pred, y_test)

print('=== Evaluation Metrics ===')
print(f'MAE  (Mean Absolute Error):       {metrics["mae"]:.4f}')
print(f'MSE  (Mean Squared Error):        {metrics["mse"]:.4f}')
print(f'RMSE (Root Mean Squared Error):   {metrics["rmse"]:.4f}')

## 6. Trend Analysis

In [ ]:
# Overall trend — is severity increasing or decreasing?
actual_trend = np.polyfit(range(len(y_test)), y_test, 1)[0]
pred_trend   = np.polyfit(range(len(y_pred)), y_pred, 1)[0]

print('=== Trend Analysis ===')
print(f'Actual trend slope:    {actual_trend:.4f}  ({"increasing" if actual_trend > 0 else "decreasing"})')
print(f'Predicted trend slope: {pred_trend:.4f}  ({"increasing" if pred_trend > 0 else "decreasing"})')

# Plot trend lines
x = np.arange(len(y_test))
actual_line = np.poly1d(np.polyfit(x, y_test, 1))
pred_line   = np.poly1d(np.polyfit(x, y_pred, 1))

plt.figure(figsize=(12, 5))
plt.scatter(x, y_test, label='Actual', alpha=0.6, color='blue')
plt.scatter(x, y_pred, label='Predicted', alpha=0.6, color='red')
plt.plot(x, actual_line(x), '--', color='blue', label='Actual Trend')
plt.plot(x, pred_line(x), '--', color='red', label='Predicted Trend')
plt.title('Severity Score Trend Analysis')
plt.xlabel('Test Sample Index')
plt.ylabel('Severity Score')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Severity score distribution comparison
plt.figure(figsize=(10, 4))
plt.hist(y_test, bins=10, alpha=0.6, label='Actual', color='blue')
plt.hist(y_pred, bins=10, alpha=0.6, label='Predicted', color='red')
plt.title('Distribution: Actual vs Predicted Severity Score')
plt.xlabel('Severity Score')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Save Outputs

In [ ]:
OUTPUT_PATH = Path('../../cleaning/data/output/forecasting_predictions.csv').resolve()
REPORT_PATH = Path('../../reports/forecasting_comparison_report.json').resolve()

# Save predictions CSV
results_df = save_predictions(y_pred, y_test, str(OUTPUT_PATH))
print(results_df.head())

# Save comparison report JSON
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
report = {
    'model': 'LSTM Forecasting Model',
    'task': 'AI013',
    'author': 'Divjot',
    'target': 'severity_score',
    'sequence_length': 5,
    'test_samples': len(y_test),
    'metrics': metrics,
    'trend': {
        'actual_slope': float(actual_trend),
        'predicted_slope': float(pred_trend),
        'direction': 'increasing' if actual_trend > 0 else 'decreasing'
    },
    'sample_predictions': results_df.head(5).to_dict(orient='records')
}

with open(REPORT_PATH, 'w') as f:
    json.dump(report, f, indent=2)

print(f'\nReport saved to: {REPORT_PATH}')

## 8. Final Summary

In [ ]:
print('=== AI013 Forecasting — Final Summary ===')
print(f'Model:              LSTM (numpy implementation)')
print(f'Target variable:    severity_score')
print(f'Sequence length:    5')
print(f'Test samples:       {len(y_test)}')
print(f'MAE:                {metrics["mae"]:.4f}')
print(f'RMSE:               {metrics["rmse"]:.4f}')
print(f'Actual trend:       {"increasing" if actual_trend > 0 else "decreasing"} ({actual_trend:.4f})')
print(f'Predicted trend:    {"increasing" if pred_trend > 0 else "decreasing"} ({pred_trend:.4f})')
print(f'Predictions saved:  {OUTPUT_PATH}')
print(f'Report saved:       {REPORT_PATH}')